In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval rapidfuzz
except Exception:
    pass

In [2]:
from dataeval import Ontology
from dataeval.core import label_alignment

In [3]:
reference = Ontology.from_hierarchy({
    "vehicle": {"car": {"sedan": None, "suv": None}, "truck": None},
    "animal": {"dog": None, "cat": None},
})
print(reference)

Ontology(8 concepts, 2 roots, 5 leaves, 0 external)


In [4]:
result = label_alignment(["sedan", "truck", "spaceship"], reference)
print("keys:", list(result))

keys: ['correspondences', 'unaligned_source', 'unaligned_target', 'class_remap', 'mergeability']


In [5]:
for c in result["correspondences"]:
    print(f"  {c.source:>10}  {c.relation:<10} {c.target:<8} ({c.confidence:.2f}, {c.matcher})")

       sedan  equivalent sedan    (1.00, exact)
       truck  equivalent truck    (1.00, exact)


In [6]:
print("class_remap:     ", dict(result["class_remap"]))
print("unaligned_source:", result["unaligned_source"])
print("mergeability:    ", result["mergeability"])

class_remap:      {'sedan': 'sedan', 'truck': 'truck'}
unaligned_source: ('spaceship',)
mergeability:     partial


In [7]:
from dataeval.core import label_reconciliation

rec = label_reconciliation(["sedan", "truck", "spaceship"], reference)
print("reconciliation matched:", dict(rec["matched"]))

reconciliation matched: {'sedan': 'sedan', 'truck': 'truck'}


In [8]:
coarse = label_alignment(["car"], reference)
for c in coarse["correspondences"]:
    print(f"  {c.source}  {c.relation:<10} {c.target}")
print("class_remap:", dict(coarse["class_remap"]), "| mergeability:", coarse["mergeability"])

  car  equivalent car
  car  broader    sedan
  car  broader    suv
class_remap: {'car': 'car'} | mergeability: lossless


In [9]:
coarse_reference = Ontology.from_hierarchy({"vehicle": {"car": None, "truck": None}})
fine_source = Ontology.from_hierarchy({"car": {"sedan": None}})

result = label_alignment(fine_source, coarse_reference)
for c in result["correspondences"]:
    print(f"  {c.source:>6}  {c.relation:<10} {c.target}")
print("class_remap:", dict(result["class_remap"]), "| mergeability:", result["mergeability"])

     car  equivalent car
   sedan  narrower   car
class_remap: {'car': 'car', 'sedan': 'car'} | mergeability: lossy


In [10]:
from collections.abc import Iterable

from rapidfuzz import fuzz, process

from dataeval.types import Correspondence, OntologyConcept


class FuzzyMatcher:
    """A Matcher proposing equivalences by fuzzy string similarity."""

    def __init__(self, threshold: float = 0.85):
        self.threshold = threshold

    # A matcher only needs to iterate concepts (id / label / synonyms) — not the
    # full Ontology — so it accepts any iterable of OntologyConcept.
    def __call__(self, source: Iterable[OntologyConcept], target: Iterable[OntologyConcept]) -> list[Correspondence]:
        # case-folded target label/synonym -> concept id
        names = {n.casefold(): c.id for c in target for n in (c.label, *c.synonyms)}
        proposals = []
        for concept in source:
            match = process.extractOne(
                concept.label.casefold(),
                list(names),
                scorer=fuzz.token_sort_ratio,
                score_cutoff=self.threshold * 100,
            )
            if match is not None:
                proposals.append(
                    Correspondence(
                        source=concept.id,
                        target=names[match[0]],
                        relation="equivalent",
                        confidence=match[1] / 100,
                        matcher="fuzzy",
                    )
                )
        return proposals

In [11]:
from dataeval.protocols import Matcher

print("is a Matcher:", isinstance(FuzzyMatcher(), Matcher))

is a Matcher: True


In [12]:
messy = ["motorcyle", "sedans", "truck pickup", "stop sign"]  # 3 near-misses, 1 OOV
catalog = Ontology.from_hierarchy({"vehicle": {"motorcycle": None, "sedan": None, "pickup truck": None}})

fuzzy = label_alignment(messy, catalog, matchers=[FuzzyMatcher(threshold=0.85)])
for c in fuzzy["correspondences"]:
    print(f"  {c.source:>12}  ->  {c.target:<14} ({c.confidence:.2f}, {c.matcher})")
print("unaligned_source:", fuzzy["unaligned_source"])

     motorcyle  ->  motorcycle     (0.95, fuzzy)
        sedans  ->  sedan          (0.91, fuzzy)
  truck pickup  ->  pickup truck   (1.00, fuzzy)
unaligned_source: ('stop sign',)


In [13]:
sources = {
    "dataset_a": ["sedan", "truck"],
    "dataset_b": ["suv", "dog"],
}
for name, labels in sources.items():
    r = label_alignment(labels, reference)
    print(f"{name:>10}: class_remap={dict(r['class_remap'])}  mergeability={r['mergeability']}")

 dataset_a: class_remap={'sedan': 'sedan', 'truck': 'truck'}  mergeability=lossless
 dataset_b: class_remap={'suv': 'suv', 'dog': 'dog'}  mergeability=lossless
